# Part 1: Genesis Franka Pick-and-Place

Headless GenesisでIK expertを収集し、後続フェーズで使うtrain/robustness用LeRobot Datasetを生成する。

In [ ]:
from pathlib import Path

from IPython.display import Video, display
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from torch.utils.data import DataLoader

from env import FRONT_IMAGE_KEY, WRIST_IMAGE_KEY, TinyPickPlaceConfig
from part1_simulator import DEFAULT_STAGE_STEPS, collect_phase1_dataset_suite

repo_id_prefix = "local/simple-vla-genesis-franka-pick-place"
dataset_root = Path("datasets/simple-vla-genesis-franka-pick-place")
image_size = 225
n_episodes = 50
stage_steps = DEFAULT_STAGE_STEPS
overwrite_dataset = True

config = TinyPickPlaceConfig(n_episodes=n_episodes, image_size=image_size, seed=0)
stage_steps


In [ ]:
datasets = collect_phase1_dataset_suite(
    root=dataset_root,
    repo_id_prefix=repo_id_prefix,
    config=config,
    n_episodes=n_episodes,
    stage_steps=stage_steps,
    image_size=image_size,
    show_viewer=False,
    backend="gpu",
    overwrite=overwrite_dataset,
)
datasets.keys()


In [ ]:
train_repo_id = f"{repo_id_prefix}-train"
train_root = dataset_root / "train"
dataset = LeRobotDataset(train_repo_id, root=train_root, video_backend="pyav")
sample = dataset[0]

print("frames:", len(dataset))
print("front:", tuple(sample[FRONT_IMAGE_KEY].shape))
print("wrist:", tuple(sample[WRIST_IMAGE_KEY].shape))
print("state:", tuple(sample["observation.state"].shape))
print("qpos:", tuple(sample["observation.robot_qpos"].shape))
print("action:", tuple(sample["action"].shape))
print("object_pose:", tuple(sample["object_pose"].shape))
print("target_pose:", tuple(sample["target_pose"].shape))
print("gripper_width:", tuple(sample["gripper_width"].shape))
print("task:", sample["task"])


In [ ]:
loader = DataLoader(dataset, batch_size=4, shuffle=False)
batch = next(iter(loader))
print(
    batch[FRONT_IMAGE_KEY].shape,
    batch[WRIST_IMAGE_KEY].shape,
    batch["observation.state"].shape,
    batch["action"].shape,
)

In [ ]:
front_video = next((train_root / "videos" / FRONT_IMAGE_KEY).glob("chunk-*/file-*.mp4"))
wrist_video = next((train_root / "videos" / WRIST_IMAGE_KEY).glob("chunk-*/file-*.mp4"))

display(Video(str(front_video), embed=True))
display(Video(str(wrist_video), embed=True))
